# Topic: Time Series Forecasting (Trend, Seasonality, Stationarity, Splits)

## Definition (30-second explanation)
Time series forecasting involves using historical, time-ordered data to predict future values. Unlike standard regression, temporal order matters heavily; you cannot shuffle the data or use future data to predict the past.

## Why Interviewers Ask This
* To verify you understand the strict rules of chronological data handling (no data leakage).
* To see if you can translate statistical requirements (like stationarity) into practical modeling steps.
* To check your business sense in translating mathematical errors into business impact metrics.

## Core Concepts
* **Trend:** The long-term upward or downward movement in the data.
* **Seasonality:** Repeating, predictable patterns at fixed intervals (e.g., weekly, annual).
* **Stationarity:** When statistical properties (mean, variance) remain constant over time; checked via ADF test.
* **Chronological Split:** Splitting train/test data strictly by time (e.g., train on 2021-2022, test on 2023) to prevent data leakage.

## When to Use
* **ARIMA:** For stationary (or easily differenced) data without strong seasonality.
* **SARIMA / Prophet:** For business data with strong seasonal patterns or holidays.
* **XGBoost / Random Forest:** For non-linear relationships, assuming heavy feature engineering (lags, rolling windows).
* **LSTM (Deep Learning):** For complex, high-dimensional, or long-sequence temporal patterns.

## Advantages
* Captures temporal dependencies that standard regression models ignore.
* Provides confidence intervals for future estimates (crucial for risk management).
* Easily interpretable baseline models (like ARIMA or Exponential Smoothing).

## Limitations
* Highly sensitive to structural breaks (e.g., COVID-19 impact on retail).
* Statistical models (ARIMA) struggle with multiple seasonalities.
* Requires continuous data; missing values need careful interpolation.

## Common Comparisons
* **ARIMA vs. Prophet:** ARIMA requires manual differencing and parameter tuning; Prophet is more robust to missing data/holidays and easier to set up out-of-the-box.
* **Statistical vs. ML (XGBoost):** Statistical models assume linear relationships and explicitly model time; ML models require you to engineer "time" into features (lags) but handle non-linearities better.

## Common Interview Traps
* **Random Splitting:** Suggesting a random 80/20 train-test split (guarantees immediate rejection).
* **Ignoring Stationarity:** Fitting ARIMA on non-stationary data without differencing.
* **Misinterpreting Metrics:** Relying solely on RMSE instead of business-friendly metrics like MAPE ("We are off by X% on average").
* **Data Leakage:** Scaling or normalizing data before the chronological split.

## Python / SQL Syntax (if applicable)
```python
    # 1. Chronological Split
    train = df[df.index < '2023-01-01']
    test = df[df.index >= '2023-01-01']

    # 2. Stationarity Test (ADF)
    from statsmodels.tsa.stattools import adfuller
    p_value = adfuller(df['sales'])[1] # If p < 0.05, data is stationary

    # 3. Differencing (if non-stationary)
    df_diff = df['sales'].diff().dropna()
```

## Important Formula (if applicable)
* **ADF Test Hypothesis:** $H_0$: Data is non-stationary. $H_a$: Data is stationary. (Reject $H_0$ if $p < 0.05$).
* **ARIMA (p, d, q):** 
    * `p`: Auto-Regressive (AR) lags
    * `d`: Degree of differencing (I)
    * `q`: Moving Average (MA) lags

## 45-Second Interview Answer
"Time series forecasting predicts future values based on historical, time-ordered data. The most critical aspect is respecting the temporal order—meaning we must use chronological train-test splits to avoid data leakage. Before applying traditional models like ARIMA, we check for stationarity using the ADF test and difference the data if needed. Finally, I always evaluate models using both statistical metrics like RMSE and business-interpretable metrics like MAPE to communicate performance to stakeholders effectively."

## Practice Questions:

### Q1: What is stationarity and why does it matter for ARIMA?
* **Answer:** Stationarity means a time series has constant statistical properties over time (constant mean, variance, and autocorrelation). It matters for ARIMA because the model's underlying mathematical assumptions require the data to be stationary to make reliable, consistent predictions. If it's not, we apply differencing (the 'I' in ARIMA) to achieve it.
* **Common Mistakes:** Confusing stationarity with seasonality, or failing to mention how to test for it (ADF test).
* **Likely Follow-up:** How do you interpret the p-value of an Augmented Dickey-Fuller (ADF) test? 
    * **Answer:** A p-value below 0.05 rejects the null hypothesis, indicating the time series is stationary.


### Q2: How do you split time series data for train/test evaluation?
* **Answer:** You must use a chronological split to respect the temporal order. For example, using the first 80% of dates for training and the most recent 20% for testing. For cross-validation, I would use rolling-window or expanding-window origin splits, ensuring the training set always precedes the validation set in time.
* **Common Mistakes:** Suggesting `sklearn.model_selection.train_test_split` with `shuffle=True`.
* **Likely Follow-up:** How would you implement cross-validation for a time series?
    * **Answer:** Use expanding or rolling windows (like `TimeSeriesSplit` in scikit-learn) where the training set strictly precedes the validation set temporally.

### Q3: What is the difference between AR, I, and MA in ARIMA?
* **Answer:** AR (Auto-Regressive) uses past values of the target variable to predict the current value. I (Integrated) represents the number of differencing steps needed to make the series stationary. MA (Moving Average) uses past forecast errors (residuals) to predict the current value.
* **Common Mistakes:** Confusing the MA component of ARIMA with calculating a simple moving average (like a 7-day rolling mean).
* **Likely Follow-up:** How do you determine the optimal values for the AR (p) and MA (q) terms?
    * **Answer:** You visually analyze the ACF and PACF plots to identify cutoffs, or run a grid search (like `auto_arima`) to minimize AIC/BIC.

### Q4: How would you forecast weekly sales for a retail store with seasonal patterns?
* **Answer:** Since the data has seasonal patterns, standard ARIMA won't suffice. I would use SARIMA to account for the seasonality explicitly. Alternatively, I would use Facebook Prophet, which is excellent for retail data with holidays, or an ML model like XGBoost, provided I create feature engineering lags (e.g., sales 7 days ago, 365 days ago) and date features (day of week, month).
* **Common Mistakes:** Recommending standard ARIMA without addressing the seasonality component.
* **Likely Follow-up:** If you use XGBoost instead of SARIMA, how do you handle the temporal nature of the data?
    * **Answer:** You must engineer "time" into explicit columns, such as lag features, rolling statistics (means/stds), and date components (day of week).

### Q5: What evaluation metrics are best for time series forecasting?
* **Answer:** It depends on the audience. For model optimization, RMSE or MAE are standard as they penalize absolute errors. However, for business stakeholders, MAPE (Mean Absolute Percentage Error) is best because it translates the error into a percentage (e.g., "our forecast is off by 5% on average"), which is much easier to digest than raw unit errors.
* **Common Mistakes:** Only mentioning RMSE and ignoring business interpretability.
* **Likely Follow-up:** What is a major drawback of using MAPE?
    * **Answer:** It becomes undefined or extremely inflated when actual values are zero or very close to zero.

### Q6: 
**You mentioned earlier that we could use an ML model like XGBoost for time series if we use feature engineering. We have a simple Pandas DataFrame containing daily sales data. Write a python function using Pandas to engineer features that would allow XGBoost to capture a weekly seasonal pattern and a recent short-term trend. Return the updated DataFrame.**

In [14]:
# Data:
import pandas as pd
import numpy as np

# Mock Schema
data = {
    'date': pd.date_range(start='2023-01-01', periods=14, freq='D'),
    'sales': [100, 120, 110, 150, 160, 200, 210, 105, 125, 115, 155, 165, 205, 215]
}
df = pd.DataFrame(data).set_index('date')

In [15]:
df

,sales
date,
2023-01-01,100
2023-01-02,120
2023-01-03,110
2023-01-04,150
2023-01-05,160
2023-01-06,200
2023-01-07,210
2023-01-08,105
2023-01-09,125


In [16]:
def engineer_ts_features(df):
    df = df.copy()
    
    # 1. Date Features (Capture explicit seasonal markers)
    df['day_of_week'] = df.index.dayofweek
    
    # 2. Lag Features (Capture what happened exactly one cycle ago)
    df['lag_7'] = df['sales'].shift(7)
    
    # 3. Rolling Window Features (Capture recent momentum/trend)
    # CRITICAL: Must shift(1) first to avoid data leakage!
    df['rolling_3d_avg'] = df['sales'].shift(1).rolling(window=3).mean()
    
    # 4. Drop NaNs created by shifting
    return df.dropna()

In [17]:
new_df= engineer_ts_features(df= df)
new_df

,sales,day_of_week,lag_7,rolling_3d_avg
date,,,,
2023-01-08,105,6,100.0,190.000000
2023-01-09,125,0,120.0,171.666667
2023-01-10,115,1,110.0,146.666667
2023-01-11,155,2,150.0,115.000000
2023-01-12,165,3,160.0,131.666667
2023-01-13,205,4,200.0,145.000000
2023-01-14,215,5,210.0,175.000000


- Common Mistakes: Calculating a rolling mean without shift(1), which includes today's sales in today's features (massive data leakage).

- Likely Follow-up: What happens to the first 7 days of your dataset after running this function? (Answer: They are dropped because lag_7 creates NaN values for the first 7 rows, meaning we lose a small amount of training data at the beginning of the series.)